# Suraksha — Phase 2 classifier training

Trains a logistic-regression head on top of multilingual sentence embeddings, plus rule-engine and metadata features. Exports the head as JSON for in-browser inference.

**Runtime → Change runtime type → T4 GPU** (free tier is fine; CPU also works, just slower).

Why embeddings and not TF-IDF: TF-IDF learns the exact tokens in the training data. A scammer swapping one Gujarati word defeats it, and the rule engine already does keyword matching — a bag-of-words model would be largely redundant with it. Embeddings generalise across paraphrase, which is the point of adding a neural layer.

In [ ]:
!pip -q install sentence-transformers scikit-learn optimum[exporters] onnx onnxruntime

## 1. Load and validate the corpus

Upload `corpus.jsonl` to the Colab file pane first.

In [ ]:
import json, re, collections
import pandas as pd

ARCHETYPES = {"kyc-expiry","upi-collect","refund-reversal","digital-arrest","courier-parcel",
  "electricity-bill","army-officer","loan-advance-fee","job-task","lottery-prize","sim-swap",
  "card-upgrade","investment-trading","qr-receive","phishing-link","other"}
TACTICS = {"urgency","authority","fear","reward","secrecy","credential","trust","irreversibility"}

rows, errors = [], []
seen_ids, seen_text = set(), set()

with open("corpus.jsonl", encoding="utf-8") as f:
    for n, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        try:
            r = json.loads(line)
        except json.JSONDecodeError as e:
            errors.append(f"line {n}: bad JSON ({e})"); continue
        for k in ("id","text","lang","label","source"):
            if k not in r:
                errors.append(f"line {n}: missing '{k}'")
        if r.get("id") in seen_ids:
            errors.append(f"line {n}: duplicate id {r['id']}")
        seen_ids.add(r.get("id"))
        norm = re.sub(r"\s+", " ", r.get("text","")).strip().lower()
        if norm in seen_text:
            errors.append(f"line {n}: duplicate text ({r.get('id')})")
        seen_text.add(norm)
        if len(r.get("text","")) < 20:
            errors.append(f"line {n}: text under 20 chars ({r.get('id')})")
        if r.get("label") == "scam":
            if r.get("archetype") not in ARCHETYPES:
                errors.append(f"line {n}: bad archetype {r.get('archetype')}")
            bad = set(r.get("tactics", [])) - TACTICS
            if bad:
                errors.append(f"line {n}: bad tactics {bad}")
        elif r.get("label") == "legit":
            if r.get("archetype") is not None:
                errors.append(f"line {n}: legit row has archetype")
        else:
            errors.append(f"line {n}: bad label {r.get('label')}")
        rows.append(r)

print(f"{len(rows)} rows, {len(errors)} problems")
for e in errors[:30]:
    print(" ", e)

df = pd.DataFrame(rows)
print()
print(pd.crosstab(df.lang, df.label))
print()
print(df[df.label=="scam"].archetype.value_counts())

**Stop here if the counts are badly skewed.** You want roughly 50/50 scam/legit within each language. A model trained on 80% scams will flag everything and score well on accuracy while being useless in the field.

## 2. Rule-engine features

Run the rule engine over the corpus **from your Next.js repo** and export per-row group scores. In the repo:

```bash
npx tsx scripts/score-corpus.ts data/corpus.jsonl > data/rule-features.json
```

See `scripts/score-corpus.ts` in the handoff prompt. Upload the resulting JSON here.

Expected shape: `{ "gu-0001": {"credential": 85, "upi": 0, "url": 70, ...}, ... }`

In [ ]:
import numpy as np

GROUPS = ["credential","upi","url","loan","impersonation","urgency","reward","textual"]

try:
    rule_feats = json.load(open("rule-features.json", encoding="utf-8"))
    have_rules = True
except FileNotFoundError:
    print("rule-features.json not found — training on embeddings + metadata only.")
    print("Add rule features before reporting final metrics; they are a big part of the ensemble story.")
    rule_feats, have_rules = {}, False

def rule_vector(row_id):
    d = rule_feats.get(row_id, {})
    return [d.get(g, 0) / 100.0 for g in GROUPS]

R = np.array([rule_vector(r["id"]) for r in rows], dtype=np.float32)
print("rule feature matrix:", R.shape)

## 3. Metadata features

Cheap, language-agnostic signals. Kept deliberately small — these must be reproducible in TypeScript at inference time, so nothing here can depend on a Python library.

In [ ]:
URL_RE = re.compile(r"(?:https?://|www\.)\S+|\b[a-z0-9-]+\.(?:com|in|xyz|top|buzz|click|ly|net|org)\b", re.I)
DIGIT_RE = re.compile(r"\d")
AMOUNT_RE = re.compile(r"(?:₹|rs\.?|inr|રૂ|रु)\s?[\d,]+", re.I)

def meta_vector(text):
    n = max(len(text), 1)
    return [
        min(len(text) / 300.0, 2.0),
        min(len(URL_RE.findall(text)) / 3.0, 1.0),
        len(DIGIT_RE.findall(text)) / n,
        min(len(AMOUNT_RE.findall(text)) / 3.0, 1.0),
        sum(1 for c in text if c.isupper()) / n,
        min(text.count("!") / 3.0, 1.0),
    ]

META_NAMES = ["len","url_count","digit_ratio","amount_count","upper_ratio","exclam"]
M = np.array([meta_vector(r["text"]) for r in rows], dtype=np.float32)
print("metadata matrix:", M.shape)

## 4. Embeddings

`paraphrase-multilingual-MiniLM-L12-v2` — 384 dims, covers Gujarati/Hindi/English, quantises to about 30 MB for the browser.

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
encoder = SentenceTransformer(MODEL_ID)

texts = [r["text"] for r in rows]
E = encoder.encode(texts, batch_size=32, show_progress_bar=True,
                   normalize_embeddings=True, convert_to_numpy=True)
print("embeddings:", E.shape)

In [ ]:
X = np.hstack([E, R, M]).astype(np.float32)
y = np.array([1 if r["label"] == "scam" else 0 for r in rows])
langs = np.array([r["lang"] for r in rows])

FEATURE_NAMES = ([f"emb_{i}" for i in range(E.shape[1])] +
                 [f"rule_{g}" for g in GROUPS] + [f"meta_{m}" for m in META_NAMES])
print("X:", X.shape, "| positives:", int(y.sum()), "| negatives:", int((1-y).sum()))

## 5. Train and evaluate

Stratified split on label **and** language, so the test set isn't accidentally all English.

`class_weight="balanced"` plus a recall-leaning threshold: a missed scam costs someone their savings, a false alarm costs ten seconds. We choose the threshold deliberately rather than defaulting to 0.5, and we say so in the README.

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, precision_recall_curve)

strata = np.array([f"{l}-{lab}" for l, lab in zip(langs, y)])
idx = np.arange(len(y))
tr, te = train_test_split(idx, test_size=0.25, random_state=42, stratify=strata)

clf = LogisticRegression(max_iter=3000, C=1.0, class_weight="balanced")
clf.fit(X[tr], y[tr])

proba = clf.predict_proba(X[te])[:, 1]
print("ROC-AUC:", round(roc_auc_score(y[te], proba), 4))

cv = cross_val_score(clf, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=0), scoring="f1")
print("5-fold F1:", cv.round(3), "| mean", round(cv.mean(), 3))

In [ ]:
# Pick the threshold: highest precision that still holds recall >= 0.95
prec, rec, thr = precision_recall_curve(y[te], proba)
TARGET_RECALL = 0.95
ok = [(p, r, t) for p, r, t in zip(prec[:-1], rec[:-1], thr) if r >= TARGET_RECALL]
THRESHOLD = max(ok, key=lambda x: x[0])[2] if ok else 0.5
print(f"threshold = {THRESHOLD:.3f} (target recall {TARGET_RECALL})")

pred = (proba >= THRESHOLD).astype(int)
print(classification_report(y[te], pred, target_names=["legit","scam"], digits=3))
cm = confusion_matrix(y[te], pred)
print("confusion matrix [rows=true, cols=pred]")
print(pd.DataFrame(cm, index=["true legit","true scam"], columns=["pred legit","pred scam"]))
print(f"\nMissed scams (the number that matters): {cm[1,0]}")

In [ ]:
# Per-language metrics — required for the README, and the first thing a juror will ask
from sklearn.metrics import precision_score, recall_score, f1_score

out = []
for lg in sorted(set(langs[te])):
    m = langs[te] == lg
    if m.sum() == 0:
        continue
    out.append({
        "lang": lg, "n": int(m.sum()),
        "precision": round(precision_score(y[te][m], pred[m], zero_division=0), 3),
        "recall": round(recall_score(y[te][m], pred[m], zero_division=0), 3),
        "f1": round(f1_score(y[te][m], pred[m], zero_division=0), 3),
    })
print(pd.DataFrame(out).to_string(index=False))

In [ ]:
# Inspect every error. These rows are where the corpus needs work.
for i, j in enumerate(te):
    if pred[i] != y[j]:
        kind = "MISSED SCAM" if y[j] == 1 else "false alarm"
        print(f"[{kind}] {rows[j]['id']} p={proba[i]:.2f}")
        print(f"   {rows[j]['text'][:160]}")
        print()

## 6. Ablation — does the ML layer actually add anything?

Run this. If rules alone already match the full model, say so honestly in the README and reconsider the weighting. A juror who asks "what does the ML give you over the rules?" needs a number, not a claim.

In [ ]:
def evaluate(Xs, name):
    c = LogisticRegression(max_iter=3000, class_weight="balanced").fit(Xs[tr], y[tr])
    p = (c.predict_proba(Xs[te])[:, 1] >= 0.5).astype(int)
    print(f"{name:28s} F1={f1_score(y[te], p):.3f}  recall={recall_score(y[te], p):.3f}  "
          f"precision={precision_score(y[te], p, zero_division=0):.3f}")

evaluate(R, "rules only")
evaluate(M, "metadata only")
evaluate(E, "embeddings only")
evaluate(np.hstack([E, M]), "embeddings + metadata")
evaluate(X, "full ensemble")

## 7. Export the head for the browser

The classifier is 398 floats plus an intercept — a few KB of JSON. The encoder ships separately as quantised ONNX.

In [ ]:
head = {
    "version": 1,
    "model_id": MODEL_ID,
    "embedding_dim": int(E.shape[1]),
    "rule_groups": GROUPS,
    "meta_features": META_NAMES,
    "coef": clf.coef_[0].astype(float).round(6).tolist(),
    "intercept": float(clf.intercept_[0]),
    "threshold": float(THRESHOLD),
    "normalize_embeddings": True,
    "trained_on": {
        "total": int(len(y)), "train": int(len(tr)), "test": int(len(te)),
        "by_lang": {k: int(v) for k, v in collections.Counter(langs).items()},
    },
}
with open("classifier-head.json", "w") as f:
    json.dump(head, f, indent=2)

import os
print("classifier-head.json:", os.path.getsize("classifier-head.json") / 1024, "KB")
print("\nInference in TS:  sigmoid(dot(coef, [...emb, ...rules, ...meta]) + intercept)")

In [ ]:
# Export the encoder as int8 ONNX for Transformers.js
!optimum-cli export onnx --model sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 --task feature-extraction onnx_out/
!python -c "from onnxruntime.quantization import quantize_dynamic, QuantType; quantize_dynamic('onnx_out/model.onnx','onnx_out/model_quantized.onnx',weight_type=QuantType.QInt8)"
!ls -la onnx_out/ | grep -E 'model.*onnx'

Check `model_quantized.onnx` is comfortably under the 40 MB model budget from §10. If it isn't, either use the Hugging Face CDN (free, no key) instead of `/public/models`, or drop to a smaller encoder.

**Sanity check before you ship:** verify the browser's embedding for a known string matches Python's to ~4 decimal places. Tokeniser and pooling differences between the two runtimes are the classic silent failure here — the model "works" but every score is subtly wrong.

In [ ]:
probe = "Your KYC is expiring today. Update now to avoid account block."
v = encoder.encode([probe], normalize_embeddings=True)[0]
print("probe:", probe)
print("first 8 dims:", np.round(v[:8], 5).tolist())
print("\nCompare these against the browser's output for the same string.")

## Download

- `classifier-head.json` → `models/` in the repo
- `onnx_out/model_quantized.onnx` → `models/` (or leave on the HF CDN)
- Paste the metrics tables into `docs/METRICS.md`, including the failures and the ablation

Commit the notebook itself. "Reproducible" is worth real marks under Technical Accuracy, and it costs nothing.